<a href="https://colab.research.google.com/github/syifaymd01/TESIS/blob/main/Copy_of_Data_Simulasi_Korelasi02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Simulasi

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import pairwise_distances, accuracy_score

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
num_iterasi = 1000
num_samples = 3000
num_variables = 8

PSO

In [ ]:
def fitness_func(dataset, centroids):
    num=len(centroids)
    distances = pairwise_distances(dataset, centroids, metric='euclidean')
    labels = np.argmin(distances, axis=1)
    if len(np.unique(labels)) < 2:
      return [0] * num
    silhouette_value = silhouette_samples(dataset, labels)
    cluster_silhouette_avg = []
    for cluster in range(num):
        cluster_silhouette_values = silhouette_value[labels == cluster]
        if len(cluster_silhouette_values) > 0:
          cluster_avg = np.mean(cluster_silhouette_values)
          cluster_silhouette_avg.append(cluster_avg)
        else:
          cluster_silhouette_avg.append(0)
    return cluster_silhouette_avg

def update_velocity(particle, velocity, pbest, gbest, w_min=0.5, maksimum=1.0, c=0.1):
  num_particle = len(particle)
  new_velocity = np.array([0 for i in range(num_particle)])
  r1 = random.uniform(0,maksimum)
  r2 = random.uniform(0,maksimum)
  w = random.uniform(w_min,maksimum)
  c1 = c
  c2 = c
  for i in range(num_particle):
    new_velocity[i] = w*velocity[i] + c1*r1*(pbest[i]-particle[i])+c2*r2*(gbest[i]-particle[i])
  return new_velocity

def update_position(particle, new_velocity):
  new_particle = particle + new_velocity
  return new_particle

def pso_km(X, num_particles, dimension, position_min, position_max, max_iterations, fitness_criterion):
  particles = np.random.uniform(position_min, position_max, (num_particles, dimension))
  pbest_position = np.copy(particles)
  pbest_fitness = np.array(fitness_func(X, particles))
  gbest_index = np.argmax(pbest_fitness)
  gbest_position = pbest_position[gbest_index]
  gbest_fitness = np.max(pbest_fitness)
  velocity = np.zeros((num_particles, dimension))

  for t in range(max_iterations):

    if np.average(pbest_fitness) <= fitness_criterion:
      break
    else:
      for n in range(num_particles):

        velocity[n] = update_velocity(particles[n], velocity[n], pbest_position[n], gbest_position)
        particles[n] = np.array(update_position(particles[n], velocity[n]))
    fitness_value = np.array(fitness_func(X, particles))
    improved_indices = np.where(fitness_value > pbest_fitness)
    pbest_fitness[improved_indices] = fitness_value[improved_indices]
    pbest_position[improved_indices] = particles[improved_indices]
    if np.max(fitness_value) > gbest_fitness:
      gbest_position = particles[np.argmax(fitness_value)]
      gbest_fitness = np.max(fitness_value)
  return particles

In [ ]:
variansi1 = 2
korelasi1 = 0.2
matriks_cov1 = np.zeros((num_variables, num_variables))
np.fill_diagonal(matriks_cov1, variansi1)
for i in range(num_variables):
    for j in range(i + 1, num_variables):
        matriks_cov1[i, j] = korelasi1 * np.sqrt(variansi1) * np.sqrt(variansi1)
        matriks_cov1[j, i] = matriks_cov1[i, j]

## Skenario Jarak Sedang

In [ ]:
mean_s1 = [6, 1, 1, 1, 1, 1, 1, 1]
mean_s2 = [1, 6, 1, 1, 1, 1, 1, 1]
mean_s3 = [1, 1, 6, 1, 1, 1, 1, 1]

### Simulasi 5%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.05
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse5 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse5)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 5%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.05
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse5 = data1 * mask

dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse5)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse5[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse5)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse5)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Sedang dengan Sparsitas 5%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.05
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse5 = data1 * mask

centroid_psokm = pso_km(X=data_sparse5, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse5)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse5)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

# Membuat legend
handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Sedang dengan Sparsitas 5%')
plt.show()

In [ ]:
dbkmeans_accuracies = []
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.05
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse5 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse5)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse5[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse5)

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse5, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
    pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse5)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100

    dbkmeans_accuracies.append(akurasi1_dbkmeans)
    psokm_accuracies.append(akurasi1_psokm)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)
mean_accuracy_psokm = np.mean(psokm_accuracies)

# Bandingkan hasil persentase ketepatan
print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")

### Simulasi 10%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.1
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse10 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse10)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 10%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.1
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse10 = data1 * mask

dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse10)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse10[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse10)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse10)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Sedang dengan Sparsitas 10%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.1
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse10 = data1 * mask

centroid_psokm = pso_km(X=data_sparse10, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse10)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse10)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Sedang dengan Sparsitas 10%')
plt.show()

In [ ]:
dbkmeans_accuracies = []
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.1
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse10 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse10)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse10[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse10)

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse10, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
    pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse10)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100

    dbkmeans_accuracies.append(akurasi1_dbkmeans)
    psokm_accuracies.append(akurasi1_psokm)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)
mean_accuracy_psokm = np.mean(psokm_accuracies)

# Bandingkan hasil persentase ketepatan
print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")

### Simulasi 25%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.25
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse25 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse25)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 25%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.25
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse25 = data1 * mask

dbscan_clustering = DBSCAN(eps=2.1, min_samples=95).fit(data_sparse25)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse25[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse25)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse25)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Sedang dengan Sparsitas 25%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.25
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse25 = data1 * mask

centroid_psokm = pso_km(X=data_sparse25, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse25)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse25)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Sedang dengan Sparsitas 25%')
plt.show()

In [ ]:
dbkmeans_accuracies = []
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.25
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse25 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=2.1, min_samples=95).fit(data_sparse25)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse25[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse25)

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse25, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
    pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse25)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100

    dbkmeans_accuracies.append(akurasi1_dbkmeans)
    psokm_accuracies.append(akurasi1_psokm)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)
mean_accuracy_psokm = np.mean(psokm_accuracies)

# Bandingkan hasil persentase ketepatan
print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")

### Simulasi 50%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.5
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse50 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse50)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 50%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.5
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse50 = data1 * mask

dbscan_clustering = DBSCAN(eps=1.6, min_samples=100).fit(data_sparse50)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse50[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse50)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse50)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Sedang dengan Sparsitas 50%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.5
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse50 = data1 * mask

centroid_psokm = pso_km(X=data_sparse50, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse50)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse50)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Sedang dengan Sparsitas 50%')
plt.show()

In [ ]:
dbkmeans_accuracies = []
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.5
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse50 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=1.6, min_samples=100).fit(data_sparse50)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse50[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse50)

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse50, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
    pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse50)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100

    dbkmeans_accuracies.append(akurasi1_dbkmeans)
    psokm_accuracies.append(akurasi1_psokm)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)
mean_accuracy_psokm = np.mean(psokm_accuracies)

# Bandingkan hasil persentase ketepatan
print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")

### Simulasi 75%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.75
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse75 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse75)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 75%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.75
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse75 = data1 * mask

dbscan_clustering = DBSCAN(eps=1.1, min_samples=100).fit(data_sparse75)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse75[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse75)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse75)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Sedang dengan Sparsitas 75%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.75
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse75 = data1 * mask

centroid_psokm = pso_km(X=data_sparse75, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse75)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse75)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

# Membuat legend
handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Sedang dengan Sparsitas 75%')
plt.show()

In [ ]:
dbkmeans_accuracies = []
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_s1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_s2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_s3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.75
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse75 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=1.1, min_samples=100).fit(data_sparse75)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse75[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse75)

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse75, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
    pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse75)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100

    dbkmeans_accuracies.append(akurasi1_dbkmeans)
    psokm_accuracies.append(akurasi1_psokm)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)
mean_accuracy_psokm = np.mean(psokm_accuracies)

# Bandingkan hasil persentase ketepatan
print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")

## Skenario Jarak Jauh

In [ ]:
mean_j1 = [20, 1, 1, 1, 1, 1, 1, 1]
mean_j2 = [1, 20, 1, 1, 1, 1, 1, 1]
mean_j3 = [1, 1, 20, 1, 1, 1, 1, 1]

### Simulasi 5%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.05
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse5 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse5)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 5%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.05
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse5 = data1 * mask

dbscan_clustering = DBSCAN(eps=2.8, min_samples=90).fit(data_sparse5)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse5[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse5)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse5)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Jauh dengan Sparsitas 5%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.05
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse5 = data1 * mask

centroid_psokm = pso_km(X=data_sparse5, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse5)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse5)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Jauh dengan Sparsitas 5%')
plt.show()

In [ ]:
dbkmeans_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.05
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse5 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=2.8, min_samples=90).fit(data_sparse5)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse5[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse5)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    dbkmeans_accuracies.append(akurasi1_dbkmeans)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)

print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")

In [ ]:
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.05
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse5 = data1 * mask

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse5, num_particles=3, dimension=8, position_min=0.1, position_max=20.1, max_iterations=100, fitness_criterion=0.01)

    if len(np.unique(np.argmin(pairwise_distances(data_sparse5, centroid_psokm), axis=1))) < 3:
        continue

    pso_kmeans = KMeans(n_clusters=3, init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse5)

    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100
    psokm_accuracies.append(akurasi1_psokm)


mean_accuracy_psokm = np.mean(psokm_accuracies)
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")

### Simulasi 10%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.1
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse10 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse10)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 10%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.1
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse10 = data1 * mask

dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse10)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse10[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse10)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse10)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Jauh dengan Sparsitas 10%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.1
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse10 = data1 * mask

centroid_psokm = pso_km(X=data_sparse10, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse10)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse10)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Jauh dengan Sparsitas 10%')
plt.show()

In [ ]:
dbkmeans_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.1
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse10 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse10)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse10[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse10)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    dbkmeans_accuracies.append(akurasi1_dbkmeans)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)

print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")

In [ ]:
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.1
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse10 = data1 * mask

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse10, num_particles=3, dimension=8, position_min=0.1, position_max=20.1, max_iterations=100, fitness_criterion=0.01)

    if len(np.unique(np.argmin(pairwise_distances(data_sparse10, centroid_psokm), axis=1))) < 3:
        continue

    pso_kmeans = KMeans(n_clusters=3, init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse10)

    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100
    psokm_accuracies.append(akurasi1_psokm)


mean_accuracy_psokm = np.mean(psokm_accuracies)
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")

### Simulasi 25%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.25
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse25 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse25)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 25%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.25
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse25 = data1 * mask

dbscan_clustering = DBSCAN(eps=2.2, min_samples=100).fit(data_sparse25)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse25[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse25)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse25)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Jauh dengan Sparsitas 25%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.25
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse25 = data1 * mask

centroid_psokm = pso_km(X=data_sparse25, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse25)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse25)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Jauh dengan Sparsitas 25%')
plt.show()

In [ ]:
dbkmeans_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.25
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse25 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=2.2, min_samples=100).fit(data_sparse25)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse25[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse25)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    dbkmeans_accuracies.append(akurasi1_dbkmeans)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)

print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")

In [ ]:
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.25
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse25 = data1 * mask

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse25, num_particles=3, dimension=8, position_min=0.1, position_max=20.1, max_iterations=100, fitness_criterion=0.01)

    if len(np.unique(np.argmin(pairwise_distances(data_sparse25, centroid_psokm), axis=1))) < 3:
        continue

    pso_kmeans = KMeans(n_clusters=3, init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse25)

    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100
    psokm_accuracies.append(akurasi1_psokm)


mean_accuracy_psokm = np.mean(psokm_accuracies)
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")

### Simulasi 50%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.5
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse50 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse50)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 50%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.5
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse50 = data1 * mask

dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse50)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse50[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse50)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse50)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Jauh dengan Sparsitas 50%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.5
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse50 = data1 * mask

centroid_psokm = pso_km(X=data_sparse50, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse50)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse50)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Jauh dengan Sparsitas 50%')
plt.show()

In [ ]:
dbkmeans_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.5
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse50 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse50)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse50[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse50)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    dbkmeans_accuracies.append(akurasi1_dbkmeans)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)

print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")

In [ ]:
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.5
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse50 = data1 * mask

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse50, num_particles=3, dimension=8, position_min=0.1, position_max=20.1, max_iterations=100, fitness_criterion=0.01)

    if len(np.unique(np.argmin(pairwise_distances(data_sparse50, centroid_psokm), axis=1))) < 3:
        continue

    pso_kmeans = KMeans(n_clusters=3, init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse50)

    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100
    psokm_accuracies.append(akurasi1_psokm)


mean_accuracy_psokm = np.mean(psokm_accuracies)
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")

### Simulasi 75%

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.75
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse75 = data1 * mask

labels = datasimulasi1['Label']
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse75)

# Membuat DataFrame hasil PCA
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])
df_pca['Label'] = labels

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=df_pca['Label'], cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Jarak Sedang dengan Sparsitas 75%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.75
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse75 = data1 * mask

dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse75)
labels_dbscan= dbscan_clustering.labels_
n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
core_samples_mask[dbscan_clustering.core_sample_indices_] = True
dbscan_centroids = []
for label in set(labels_dbscan):
    if label != -1:
        dbscan_points = data_sparse75[labels_dbscan == label]
        dbscan_centroid = np.mean(dbscan_points, axis=0)
        dbscan_centroids.append(dbscan_centroid)
dbscan_centroids = np.array(dbscan_centroids)
db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
labels_dbkmeans = db_kmeans.fit_predict(data_sparse75)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse75)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_dbkmeans, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_dbkmeans)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('DB-Kmeans pada Jarak Jauh dengan Sparsitas 75%')
plt.show()

In [ ]:
gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
df_gerombol1['Label'] = 0
df_gerombol2['Label'] = 1
df_gerombol3['Label'] = 2
datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

#Data Sparse
persentase_sparse = 0.75
mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
data_sparse75 = data1 * mask

centroid_psokm = pso_km(X=data_sparse75, num_particles=3, dimension=8, position_min=0, position_max=0.1, max_iterations=100, fitness_criterion=0.001)
pso_kmeans = KMeans(n_clusters=len(centroid_psokm), init=centroid_psokm, n_init=1)
labels_psokm = pso_kmeans.fit_predict(data_sparse75)

# Reduksi Dimensi
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_sparse75)
df_pca = pd.DataFrame(data_pca, columns=['Dim1', 'Dim2'])

# Scatter plot hasil PCA
plt.figure(figsize=(10, 10))
scatter = plt.scatter(df_pca['Dim1'], df_pca['Dim2'], c=labels_psokm, cmap='viridis')

handles, _ = scatter.legend_elements(prop="colors")
labels_unique = np.unique(labels_psokm)
plt.legend(handles, labels_unique, title="Clusters")

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('PSO-Kmeans pada Jarak Jauh dengan Sparsitas 75%')
plt.show()

In [ ]:
dbkmeans_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.75
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse75 = data1 * mask

    #DB-Kmeans
    dbscan_clustering = DBSCAN(eps=2.5, min_samples=100).fit(data_sparse75)
    labels_dbscan= dbscan_clustering.labels_
    n_cluster_db=len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
    core_samples_mask = np.zeros_like(dbscan_clustering.labels_, dtype=bool)
    core_samples_mask[dbscan_clustering.core_sample_indices_] = True
    dbscan_centroids = []
    for label in set(labels_dbscan):
        if label != -1:
            dbscan_points = data_sparse75[labels_dbscan == label]
            dbscan_centroid = np.mean(dbscan_points, axis=0)
            dbscan_centroids.append(dbscan_centroid)
    dbscan_centroids = np.array(dbscan_centroids)
    db_kmeans = KMeans(n_clusters=len(dbscan_centroids), init=dbscan_centroids, n_init=1)
    labels_dbkmeans = db_kmeans.fit_predict(data_sparse75)

    akurasi1_dbkmeans = accuracy_score(datasimulasi1['Label'], labels_dbkmeans)*100
    dbkmeans_accuracies.append(akurasi1_dbkmeans)

mean_accuracy_dbkm = np.mean(dbkmeans_accuracies)

print(f"Rata-rata Persentase Ketepatan DB-Kmeans: {mean_accuracy_dbkm:.2f}%")

In [ ]:
psokm_accuracies = []

for _ in range(num_iterasi):
    gerombol1 = np.random.multivariate_normal(mean_j1, matriks_cov1, num_samples)
    gerombol2 = np.random.multivariate_normal(mean_j2, matriks_cov1, num_samples)
    gerombol3 = np.random.multivariate_normal(mean_j3, matriks_cov1, num_samples)
    df_gerombol1 = pd.DataFrame(gerombol1, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol2 = pd.DataFrame(gerombol2, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol3 = pd.DataFrame(gerombol3, columns=['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8'])
    df_gerombol1['Label'] = 0
    df_gerombol2['Label'] = 1
    df_gerombol3['Label'] = 2
    datasimulasi1 = pd.concat([df_gerombol1, df_gerombol2, df_gerombol3], ignore_index=True)
    data1 = datasimulasi1.filter(["X1", "X2", "X3", "X4", "X5", "X6", "X7", "X8"])

    #Data Sparse
    persentase_sparse = 0.75
    mask = np.random.choice([0, 1], size=data1.shape, p=[persentase_sparse, 1 - persentase_sparse])
    data_sparse75 = data1 * mask

    #PSO-Kmeans
    centroid_psokm = pso_km(X=data_sparse75, num_particles=3, dimension=8, position_min=0.1, position_max=20.1, max_iterations=100, fitness_criterion=0.01)

    if len(np.unique(np.argmin(pairwise_distances(data_sparse75, centroid_psokm), axis=1))) < 3:
        continue

    pso_kmeans = KMeans(n_clusters=3, init=centroid_psokm, n_init=1)
    labels_psokm = pso_kmeans.fit_predict(data_sparse75)

    akurasi1_psokm = accuracy_score(datasimulasi1['Label'], labels_psokm)*100
    psokm_accuracies.append(akurasi1_psokm)


mean_accuracy_psokm = np.mean(psokm_accuracies)
print(f"Rata-rata Persentase Ketepatan PSO-Kmeans: {mean_accuracy_psokm:.2f}%")